# Data Validation & Cleaning

Before any analysis or modeling, it is critical to ensure that the data used in this project
is reliable, consistent, and aligned with business rules.

In this notebook, we:
- Load all source datasets
- Validate key relationships
- Identify and correct data quality issues
- Prepare clean datasets for downstream analysis

In [1]:
# Basic libraries (used across notebooks)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
plt.style.use("default")

## Load Datasets

The project uses four primary datasets:
- Sales transactions
- Inventory snapshots
- Product master
- Supplier master


In [ ]:
sales_df = pd.read_csv("/data/raw/sales_fact.csv")
inventory_df = pd.read_csv("/data/raw/inventory_snapshot.csv")
products_df = pd.read_csv("/data/raw/products_master.csv")
suppliers_df = pd.read_csv("/data/raw/suppliers_master.csv")

print("Sales shape:", sales_df.shape)
print("Inventory shape:", inventory_df.shape)
print("Products shape:", products_df.shape)
print("Suppliers shape:", suppliers_df.shape)


## Initial Data Overview

We first inspect the structure, column names, and data types of each dataset
to ensure they match the definitions provided in the data dictionary.


In [ ]:
sales_df.head()
inventory_df.head()
products_df.head()
suppliers_df.head()

## Date Validation

Dates must be correctly parsed and sorted chronologically.
Incorrect date formats or unordered timestamps can break time-series analysis.

In [ ]:
sales_df["date"] = pd.to_datetime(sales_df["date"])
inventory_df["snapshot_date"] = pd.to_datetime(inventory_df["snapshot_date"])

sales_df = sales_df.sort_values(["sku_id", "date"])
inventory_df = inventory_df.sort_values(["sku_id", "snapshot_date"])

## Key Consistency Checks

We validate whether all SKUs and suppliers referenced in transactional data
exist in the corresponding master datasets.

In [ ]:
sales_skus = set(sales_df["sku_id"].unique())
inventory_skus = set(inventory_df["sku_id"].unique())
product_skus = set(products_df["sku_id"].unique())

missing_sales_skus = sales_skus - product_skus
missing_inventory_skus = inventory_skus - product_skus

print("SKUs in sales but missing in product master:", len(missing_sales_skus))
print("SKUs in inventory but missing in product master:", len(missing_inventory_skus))

## Supplier Consistency Check

In [ ]:
supplier_ids_products = set(products_df["supplier_id"].unique())
supplier_ids_master = set(suppliers_df["supplier_id"].unique())

missing_suppliers = supplier_ids_products - supplier_ids_master
print("Suppliers missing in supplier master:", len(missing_suppliers))

## Missing Value Analysis

Missing values can represent:
- Data collection gaps
- Operational issues
- Incorrect joins

We identify and treat them carefully.

In [ ]:
def missing_summary(df):
    return df.isna().sum().sort_values(ascending=False)

missing_summary(sales_df)

In [ ]:
missing_summary(inventory_df)
missing_summary(products_df)
missing_summary(suppliers_df)

## Business Rule Checks

We now validate logical rules based on how inventory systems work.

In [ ]:
# Negative sales
negative_sales = sales_df[sales_df["units_sold"] < 0]
print("Negative sales records:", negative_sales.shape[0])

# Negative inventory
negative_inventory = inventory_df[inventory_df["on_hand_qty"] < 0]
print("Negative inventory records:", negative_inventory.shape[0])

# Invalid lead times
invalid_lead_time = suppliers_df[suppliers_df["lead_time_days"] <= 0]
print("Invalid lead time records:", invalid_lead_time.shape[0])

## Sales with Zero Inventory Check

Sales recorded when on-hand inventory is zero may indicate stockouts,
backorders, or delayed inventory updates.

In [ ]:
inventory_zero = inventory_df[inventory_df["on_hand_qty"] == 0][
    ["sku_id", "snapshot_date"]
]

sales_with_zero_inventory = sales_df.merge(
    inventory_zero,
    left_on=["sku_id", "date"],
    right_on=["sku_id", "snapshot_date"],
    how="inner"
)

print("Potential stockout-related sales records:", sales_with_zero_inventory.shape[0])

## Data Corrections & Assumptions

Based on the validation checks, the following actions are taken:

- Negative sales values are treated as data errors and removed
- Negative inventory values are clipped to zero
- Missing lead times are filled using supplier-level averages
- Records with unresolved SKU or supplier references are excluded

All assumptions are documented to maintain transparency.

In [ ]:
# Remove negative sales
sales_df = sales_df[sales_df["units_sold"] >= 0]

# Clip negative inventory
inventory_df["on_hand_qty"] = inventory_df["on_hand_qty"].clip(lower=0)

# Fill missing lead times
suppliers_df["lead_time_days"] = suppliers_df["lead_time_days"].fillna(
    suppliers_df["lead_time_days"].median()
)

## Save Cleaned Datasets

The cleaned datasets will be used in all downstream notebooks.

In [ ]:
sales_df.to_csv("data/processed/sales_clean.csv", index=False)
inventory_df.to_csv("data/processed/inventory_clean.csv", index=False)
products_df.to_csv("data/processed/products_clean.csv", index=False)
suppliers_df.to_csv("data/processed/suppliers_clean.csv", index=False)